Reference:

1. [LangSmith Docs](https://docs.smith.langchain.com/?ref=blog.langchain.dev)


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_community.llms import HuggingFaceHub

In [ ]:
repo_id = "mistralai/Mistral-7B-v0.1"

llm = HuggingFaceHub(repo_id=repo_id, model_kwargs={"temperature": 0.1})

In [ ]:
llm.invoke("What is google?")

In [ ]:
# Let's train the model on excel dataset
# pip install unstructured[xlsx]
from langchain_community.document_loaders import UnstructuredExcelLoader

In [ ]:
loader = UnstructuredExcelLoader(
    "Healthcare_insurance_excel_dataset.xlsx", mode="elements"
)

In [ ]:
docs = loader.load()
docs[0]

In [ ]:
%pip install sentence-transformers

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(docs)

In [ ]:
texts[0]

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Equivalent to SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
%pip install chromadb

In [ ]:
from langchain.vectorstores import Chroma

vector_db = Chroma.from_documents(texts, embeddings)

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorDB.as_retriever(),
    verbose=True,
    chain_type_kwargs={"verbose": True, "prompt": PROMPT},
)

In [ ]:
# from langchain.prompts import PromptTemplate

In [ ]:
# prompt_template = """You are a Insurance customer support agent.
#         Recommend the customer with the best plan suitable for the Customer's based on their payment_type, and premium amount.
#         Use the following customer related information (delimited by <cp></cp>) context (delimited by <ctx></ctx>) and the chat history (delimited by <hs></hs>) to answer the question at the end:
#         If you don't know the answer, just say that you don't know, don't try to make up an answer.
#         Below are the details of the customer:\n 
#         <cp>
#         Premium Amount: {premium_amount}
#         Payment Type: {payment_type}
#         Inpatient Care: {inpatient_care}
#         Outpatient Care: {outpatient_care}
#         Prescription Care: {prescription_drugs}
#         Mental Health Care: {mental_health_care}
#         Dental Care: {dental_care}
#         </cp>
#         <ctx>
#         {context}
#         </ctx>
#         Question: {query}
#         Answer: """

In [ ]:
# PROMPT = PromptTemplate(
#     template=prompt_template,
#     input_variables=[
#         "context",
#         "query",
#         "premium_amount",
#         "payment_type",
#         "inpatient_care",
#         "outpatient_care",
#         "prescription_drugs",
#         "mental_health_care",
#         "dental_care",
#     ],
# )

In [ ]:
# from langchain.chains import RetrievalQA

In [ ]:
# qa = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=vectorDB.as_retriever(),
#     verbose=True,
#     chain_type_kwargs={"verbose": True, "prompt": PROMPT},
# )

In [ ]:
%pip install langchain_experimental

In [ ]:
%pip install pandas

In [ ]:
import pandas as pd

In [ ]:
%pip install openpyxl

In [ ]:
all_sheet_name = pd.ExcelFile("Healthcare_insurance_excel_dataset.xlsx").sheet_names

In [ ]:
all_sheet_name

In [ ]:
# Iterate over the LLM
insurance_dfs = []
for sheet_name in all_sheet_name:
    dfs = pd.read_excel("Healthcare_insurance_excel_dataset.xlsx", sheet_name=sheet_name, engine='openpyxl')
    insurance_dfs.append(dfs)

In [ ]:
insurance_dfs

In [ ]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

In [ ]:
%pip install google-generativeai

In [ ]:
%pip install google-generativeai

In [ ]:
import os
from langchain_community.llms.google_palm import GooglePalm

llm = GooglePalm(
    google_api_key=os.environ.get("GOOGLE_API_KEY"), temperature=0
)

In [ ]:
prefix = """
You are an Insurance expert that helps customers in analysing the insurance data.
Here is the schema of the data

df1 = agents related data (For agents regarding questions refer this df)
df2 = claims related data
df3 = customers related data
df4 = payment related data
df5 = plans related data
df6 = ploicies related data
df7 = provider related data
df8 = reimbursement related data
df9 = reject_claims data
df10 = Subscribers data

Carefully identify the queries and based on that refer the dataframe to get the data.
There can be questions where we required data from multiple dfs, in that case merge the dfs 
and fetch the output

"""

In [ ]:
%pip install tabulate

In [ ]:
agent = create_pandas_dataframe_agent(llm, 
                                      insurance_dfs, 
                                      verbose=True,
                                      handle_parsing_errors=True,
                                      prefix=prefix)

In [ ]:
agent.run("How many agent ids are there in agents sheet?")

In [ ]:
agent.run("How many claims are there in claims sheet?")

In [ ]:
agent.run("How many policies are there in total?")

In [ ]:
agent.run("Cusomer Yasmin Borde has taken how many policies?")

Great, it gave the accurate output

In [ ]:
agent.run("Provider Anvi Ahuja has how many claims?")

In [ ]:
# Let me ask the complex questions
insurance_dfs[1].loc[insurance_dfs[1]['provider_id'] == 'Provider_0001'].shape[0]

In [ ]:
agent.run("How many claims are rejected due to Lack of Medical Necessity")

In [ ]:
insurance_dfs[8].merge(insurance_dfs[1], on='claim_number', how='inner').query('reason == "Lack of Medical Necessity"').count()

The output is not correct. Let's try again

**Okay, do one thing, let use the GoogleGenerativeAI**

In [ ]:
%pip install langchain_google_genai

In [ ]:
from langchain_google_genai import GoogleGenerativeAI

google_genai = GoogleGenerativeAI(model="gemini-pro")

In [ ]:
gemini_agent = agent = create_pandas_dataframe_agent(google_genai, 
                                      insurance_dfs, 
                                      verbose=True,
                                      handle_parsing_errors=True,
                                      prefix=prefix)

In [ ]:
gemini_agent.run("How many claims are rejected due to Lack of Medical Necessity")

In [ ]:
agent.run("Provider Anvi Ahuja has accessed how many claims?")

In [ ]:
agent.run("Provider Uthkarsh Cherian has accessed how many claims?")

In [ ]:
# let's ask the complex queries
gemini_agent.run(
    """Which provider have accessed the highest number of claims, and can we list them in descending order 
    based on claim access frequency?"""
)

In [ ]:
# let's ask the complex queries
gemini_agent.run(
    """Which provider have accessed the highest number of claims, and can we list them in descending order 
    based on claim access frequency?"""
)

In [ ]:
gemini_agent.run(
    """Give the name of provider who have accessed the highest number of claims based on claim access frequency?"""
)

In [ ]:
gemini_agent.run("""
    What is the average number of claims accessed by providers in the dataset, and how does 
    Uthkarsh Cherian's claim access compare to this average?
""")

Like this we can ask multiple questions and we can see the gemini-pro is giving excellent results